# Baseline Model

Train a vanilla baseline CatBoost model using only the main Home Credit application table.

No bureau, previous application, POS, installment, credit-card tables, or engineered feature files are used here. The goal is a clean baseline against the full project pipeline. Artifacts are saved to `Models/baseline/`.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import yaml

from catboost import CatBoostClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split


In [ ]:
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name.lower() == "notebooks" else cwd
config_path = project_root / "config.yaml"

with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

raw_paths = config["data"]["raw"]
target_col = config["training"]["target_col"]
id_col = config["training"]["id_col"]
seed = config["globals"]["random_state"]
baseline_config = config["baseline"]
baseline_paths = baseline_config["artifact_paths"]

train_path = project_root / raw_paths["application_train"]
test_path = project_root / raw_paths["application_test"]
baseline_dir = project_root / config["training"]["artifact_paths"]["models_dir"] / baseline_config["artifact_subdir"]
baseline_dir.mkdir(parents=True, exist_ok=True)

print(f"Train path: {train_path}")
print(f"Test path: {test_path}")
print(f"Artifacts: {baseline_dir}")


## Load Vanilla Application Tables

Only `application_train.csv` and `application_test.csv` are loaded. The notebook drops `TARGET` and `SK_ID_CURR` for modeling, but does not merge other tables or create project pipeline features.

In [ ]:
application_train = pd.read_csv(train_path)
application_test = pd.read_csv(test_path)

if target_col not in application_train.columns:
    raise ValueError(f"{target_col} not found in application_train.csv")
if target_col in application_test.columns:
    raise ValueError(f"{target_col} should not be present in application_test.csv")

y = application_train[target_col]
X = application_train.drop(columns=[target_col, id_col])
X_test_final = application_test.drop(columns=[id_col])
test_ids = application_test[id_col]

print("Train features:", X.shape)
print("Test features:", X_test_final.shape)
print("Default rate:", y.mean())


In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

for col in categorical_cols:
    X[col] = X[col].astype("string").fillna(baseline_config["categorical_missing_label"])
    X_test_final[col] = X_test_final[col].astype("string").fillna(baseline_config["categorical_missing_label"])

cat_features = [X.columns.get_loc(col) for col in categorical_cols]
print(f"Categorical columns: {len(categorical_cols)}")


## Train/Validation Split

The validation score is a simple stratified holdout baseline. It is not the full project's OOF validation.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=baseline_config["validation_size"],
    stratify=y,
    random_state=seed,
)

model_config = next(
    candidate for candidate in config["training"]["models"]["candidates"]
    if candidate["name"] == "catboost"
)
model_params = model_config["params"].copy()
early_stopping_rounds = model_params.pop("early_stopping_rounds", None)
model_params.pop("eval_fraction", None)
model_params.update(config["training"]["preprocessing"]["imbalance"]["class_weight_params"].get("catboost", {}))

validation_model = CatBoostClassifier(
    cat_features=cat_features,
    eval_metric="AUC",
    random_seed=seed,
    verbose=baseline_config["verbose"],
    **model_params,
)

fit_kwargs = {"eval_set": (X_valid, y_valid)}
if early_stopping_rounds:
    fit_kwargs["early_stopping_rounds"] = early_stopping_rounds

validation_model.fit(X_train, y_train, **fit_kwargs)


In [ ]:
valid_proba = validation_model.predict_proba(X_valid)[:, 1]
threshold = float(baseline_config["classification_threshold"])
valid_pred = (valid_proba >= threshold).astype(int)

metrics = {
    "scope": "vanilla_application_holdout",
    "roc_auc": float(roc_auc_score(y_valid, valid_proba)),
    "average_precision": float(average_precision_score(y_valid, valid_proba)),
    "classification_threshold": threshold,
    "train_rows": int(len(X_train)),
    "valid_rows": int(len(X_valid)),
    "feature_count": int(X.shape[1]),
    "categorical_feature_count": int(len(categorical_cols)),
}

report = classification_report(y_valid, valid_pred, zero_division=0)
conf_matrix = confusion_matrix(y_valid, valid_pred)

print(json.dumps(metrics, indent=2))
print(report)


## Fit Final Baseline And Save Artifacts

The final baseline model is fit on the full vanilla application training table. The validation model's best iteration is reused when available.

In [ ]:
final_params = model_params.copy()
best_iteration = validation_model.get_best_iteration()
if best_iteration is not None and best_iteration > 0:
    final_params["iterations"] = best_iteration + 1

final_model = CatBoostClassifier(
    cat_features=cat_features,
    eval_metric="AUC",
    random_seed=seed,
    verbose=baseline_config["verbose"],
    **final_params,
)
final_model.fit(X, y)

submission = pd.DataFrame({
    id_col: test_ids,
    target_col: final_model.predict_proba(X_test_final)[:, 1],
})

joblib.dump(final_model, baseline_dir / baseline_paths["final_model"])
submission.to_csv(baseline_dir / baseline_paths["submission"], index=False)

with open(baseline_dir / baseline_paths["metrics"], "w", encoding="utf-8") as file:
    yaml.safe_dump(metrics, file, sort_keys=False)

with open(baseline_dir / baseline_paths["evaluation_report"], "w", encoding="utf-8") as file:
    file.write("Baseline CatBoost on raw application table only\n")
    file.write("=" * 48 + "\n")
    file.write(json.dumps(metrics, indent=2))
    file.write(f"\n\nClassification report @ {threshold:.4f} threshold\n")
    file.write(report)

with open(baseline_dir / baseline_paths["config_snapshot"], "w", encoding="utf-8") as file:
    yaml.safe_dump(config, file, sort_keys=False)

print(f"Saved baseline artifacts to {baseline_dir}")


In [ ]:
fig, ax = plt.subplots(figsize=tuple(baseline_config["roc_figsize"]))
RocCurveDisplay.from_predictions(y_valid, valid_proba, ax=ax)
ax.set_title("Baseline CatBoost ROC Curve")
fig.tight_layout()
fig.savefig(baseline_dir / baseline_paths["roc_curve"], dpi=baseline_config["plot_dpi"])
plt.show()

fig, ax = plt.subplots(figsize=tuple(baseline_config["confusion_matrix_figsize"]))
ConfusionMatrixDisplay(confusion_matrix=conf_matrix).plot(ax=ax, colorbar=False)
ax.set_title("Baseline CatBoost Confusion Matrix")
fig.tight_layout()
fig.savefig(baseline_dir / baseline_paths["confusion_matrix"], dpi=baseline_config["plot_dpi"])
plt.show()


In [ ]:
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": final_model.get_feature_importance(),
}).sort_values("importance", ascending=False)
feature_importance.to_csv(baseline_dir / baseline_paths["feature_importance"], index=False)

top = feature_importance.head(int(baseline_config["top_feature_count"])).sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=tuple(baseline_config["feature_importance_figsize"]))
ax.barh(top["feature"], top["importance"])
ax.set_title("Baseline CatBoost Top Feature Importances")
ax.set_xlabel("Importance")
fig.tight_layout()
fig.savefig(baseline_dir / baseline_paths["feature_importance_plot"], dpi=baseline_config["plot_dpi"])
plt.show()

feature_importance.head(20)
